# OrbitLite heuristic + ML shot validator

This notebook trains a lightweight validator on top of the OrbitLite heuristic agent. It now uses Kaggle-style dataset slugs, bundles the sibling `orbit_lite/` helper package, and includes cells for feature importance, top-k experiments, and pure heuristic vs validator benchmarking.


## 0. Dataset layout

Attach two Kaggle datasets before running:

1. **Agent dataset**: contains `main.py` and sibling helper folder `orbit_lite/`.
2. **Opponent dataset**: contains opponent `.py` files used for self-play and evaluation.

Kaggle mounts datasets at `/kaggle/input/<slug>`. Edit `BASE_DATA_SLUG` and `OPPONENTS_DATA_SLUG` below to match the dataset slugs shown in the notebook sidebar.


In [ ]:
# Locate OrbitLite source and helper package.
import os
import shutil
import sys
from pathlib import Path

# Kaggle dataset slugs. Kaggle paths are /kaggle/input/<slug>/...
BASE_DATA_SLUG = os.environ.get("ORBITLITE_BASE_DATA_SLUG", "orbitlite")
OPPONENTS_DATA_SLUG = os.environ.get("ORBITLITE_OPPONENTS_DATA_SLUG", "orbit-wars-opponents")

BASE_INPUT = Path(f"/kaggle/input/{BASE_DATA_SLUG}")
OPPONENTS_INPUT = Path(f"/kaggle/input/{OPPONENTS_DATA_SLUG}")
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

# Local fallbacks make the notebook easier to test outside Kaggle.
V4_CANDIDATES = [BASE_INPUT / "main.py", Path("main.py"), Path("/kaggle/working/main.py")]
HELPER_CANDIDATES = [BASE_INPUT / "orbit_lite", Path("orbit_lite"), Path("/kaggle/working/orbit_lite")]
V4_PATH = next((p for p in V4_CANDIDATES if p.exists()), None)
HELPER_SRC = next((p for p in HELPER_CANDIDATES if p.exists() and p.is_dir()), None)

assert V4_PATH is not None, (
    f"OrbitLite main.py not found. Attach dataset slug {BASE_DATA_SLUG!r} "
    "with main.py at /kaggle/input/<slug>/main.py, or place main.py in the working directory."
)
assert HELPER_SRC is not None, (
    f"orbit_lite helper folder not found. Attach dataset slug {BASE_DATA_SLUG!r} "
    "with orbit_lite/ at /kaggle/input/<slug>/orbit_lite."
)

# Copy to working dir so generated agents and Kaggle submission can import orbit_lite.
V4_LOCAL = WORK / "baseline_main.py"
HELPER_LOCAL = WORK / "orbit_lite"
shutil.copy(V4_PATH, V4_LOCAL)
if HELPER_LOCAL.exists():
    shutil.rmtree(HELPER_LOCAL)
shutil.copytree(HELPER_SRC, HELPER_LOCAL)

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

print(f"base slug: {BASE_DATA_SLUG} -> {BASE_INPUT}")
print(f"opponents slug: {OPPONENTS_DATA_SLUG} -> {OPPONENTS_INPUT}")
print(f"base loaded: {V4_LOCAL} ({V4_LOCAL.stat().st_size:,} bytes)")
print(f"helper copied: {HELPER_LOCAL} ({len(list(HELPER_LOCAL.glob('*.py')))} python files)")


## 1. Environment setup

The `orbit_wars` environment was added in `kaggle-environments` 1.28+, but the Kaggle default image ships an older version. **Competition-linked notebooks also have internet disabled**, so we install the wheel offline from the bundled dataset.

In [ ]:
# Kaggle's preinstalled kaggle_environments may not include orbit_wars,
# and competition-linked notebooks have internet disabled. Install from an
# attached wheel if one is present.
import sys, subprocess, glob, os as _os

WHEEL_CANDIDATES = []
WHEEL_CANDIDATES.extend(glob.glob(str(BASE_INPUT / "kaggle_environments-*.whl")))
WHEEL_CANDIDATES.extend(glob.glob("/kaggle/input/orbit-wars-v4-base/kaggle_environments-*.whl"))
WHEEL_CANDIDATES.extend(glob.glob("/kaggle/input/*/kaggle_environments-*.whl"))

if WHEEL_CANDIDATES:
    wheel = sorted(WHEEL_CANDIDATES)[0]
    print(f"Installing {wheel}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", wheel], check=True)
else:
    print("No offline kaggle_environments wheel found; using the preinstalled package.")

import importlib, kaggle_environments
importlib.reload(kaggle_environments)
envs = sorted(_os.listdir(_os.path.join(_os.path.dirname(kaggle_environments.__file__), "envs")))
print(f"kaggle_environments {kaggle_environments.__version__}; orbit_wars available: {'orbit_wars' in envs}")
assert "orbit_wars" in envs, f"orbit_wars not in envs: {envs}"

import torch
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")


## 2. Define baseline opponents for self-play

We need opponents to generate training data. The original `exp_arch_001` pipeline uses 5 stronger agents (`v1_sniper`, `v2_structured`, `exp007_tier3/4`, `orbitbotnext`), but for a self-contained public notebook we use 5 simple agents written inline:

- `nearest_sniper` — attack the nearest non-allied planet (the official sample agent)
- `weakest_first` — attack the planet with the fewest defenders
- `production_first` — prioritize high-production planets
- `defender` — consolidate ships toward your own planets
- `random_play` — purely random shots (negative-class generator)

A stronger opponent pool produces a stronger validator. If you have access to better agents, swap them in by replacing the entries in `OPPONENT_CODES` below or pointing `OPPONENT_PATHS` at external files.

In [ ]:
from pathlib import Path

OPPONENTS_DIR = OPPONENTS_INPUT
assert OPPONENTS_DIR.exists(), (
    f"Opponent dataset not found at {OPPONENTS_DIR}. "
    "Edit OPPONENTS_DATA_SLUG in the setup cell."
)

OPPONENT_ALLOWLIST = {
    "h3b1.py", "hellburner.py", "launch_safety_1039.py", "orrbitnext.py",
    "aidan_song.py", "suneet.py", "exp30.py", "top_now.py", "top_now_v1.py",
    "buddy.py", "h3p1_safety.py", "multi-focus.py",
}
all_opponent_files = sorted(OPPONENTS_DIR.glob("*.py"))
OPPONENT_PATHS = [str(p) for p in all_opponent_files if p.name in OPPONENT_ALLOWLIST]
if not OPPONENT_PATHS:
    OPPONENT_PATHS = [str(p) for p in all_opponent_files]

print(f"Found {len(OPPONENT_PATHS)} opponents in {OPPONENTS_DIR}:")
for p in OPPONENT_PATHS:
    print(" ", p)

INCLUDE_SELFPLAY = True
if INCLUDE_SELFPLAY:
    OPPONENT_PATHS.append(str(V4_LOCAL))
    print(f"added OrbitLite self-play: {V4_LOCAL}")


## 3. Data collection (self-play)

We run v4 against 5 opponents × 6 seeds × 2 sides = **60 games**. Every move v4 issues is recorded as a *shot* with three pieces:

- **Source** = `(src_id, ang, ships)` — the move v4 picked
- **Features** = a 24-dim vector describing the board state at the moment of the shot (next section)
- **Label** = was this shot ultimately **successful**? Specifically: did v4 own the target planet within 10 turns of the projected fleet arrival?

So the validator learns a binary classifier: **"will this attack actually capture the target?"**

**Time budget**: roughly 5–10 minutes on Kaggle CPU.

In [ ]:
# Data collection
import math, time, multiprocessing as mp
import numpy as np
from kaggle_environments import make

BOARD = 100.0; MAX_SPEED = 6.0

def fleet_speed(ships):
    if ships <= 0: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (math.log(max(ships, 1)) / math.log(1000.0)) ** 1.5

def find_target_via_ray(src_xy, send_angle, planets, ray_horizon=200.0, perp_margin=1.0):
    """Recover the (likely) target planet of a shot from its (src, angle).

    v4 emits actions as (src_id, angle, ships) — the target is implicit. We
    project a ray from src along `angle` and find the closest planet whose
    bounding circle the ray crosses.
    """
    sx, sy = src_xy; fx, fy = math.cos(send_angle), math.sin(send_angle)
    best_pid, best_perp = -1, 1e9
    for p in planets:
        pid, _, px, py, pr, _, _ = p
        pid = int(pid); px = float(px); py = float(py); pr = float(pr)
        dx = px - sx; dy = py - sy
        t = dx * fx + dy * fy
        if t <= 0 or t > ray_horizon: continue
        perp = abs(dx * fy - dy * fx)
        if perp <= pr + perp_margin and perp < best_perp:
            best_perp = perp; best_pid = pid
    return best_pid

def label_outcome(env_steps, target_id, side, arrival_turn, window=10):
    """Label = 1 iff `side` owns `target_id` at any turn in [arrival_turn, arrival_turn+window]."""
    end_t = min(arrival_turn + window, len(env_steps) - 1)
    start_t = min(arrival_turn, end_t)
    for t in range(start_t, end_t + 1):
        s = env_steps[t][side].observation
        if s is None: continue
        for p in s["planets"]:
            if int(p[0]) == target_id and int(p[1]) == side: return 1
    return 0

# 24-dim hand-crafted feature extractor for a single shot.
FEATURE_NAMES = [
    "src_ships", "src_prod", "src_radius",
    "tgt_ships", "tgt_prod", "tgt_radius",
    "tgt_is_self", "tgt_is_neutral", "tgt_is_enemy",
    "ships_sent", "ship_frac", "dist", "eta", "speed",
    "ally_fleet_n", "ally_fleet_ships", "enemy_fleet_n", "enemy_fleet_ships",
    "turn", "my_ships_total", "enemy_ships_total", "ship_delta",
    "my_planets", "enemy_planets",
]
FEATURE_DIM = len(FEATURE_NAMES)
def encode_shot(obs, src_id, target_id, ships_sent):
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict: return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_ships_total = sum(int(p[5]) for p in planets if int(p[1]) == me)
    enemy_ships_total = sum(int(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_planets = sum(1 for p in planets if int(p[1]) == me)
    enemy_planets = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(math.hypot(dx, dy) - sr - tr, 0.0)
    speed = fleet_speed(ships_sent)
    eta = dist / max(speed, 0.5)
    own_self = 1.0 if int(tgt[1]) == me else 0.0
    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    ship_frac = ships_sent / max(sships, 1)
    ally_n = sum(1 for f in fleets if int(f[1]) == me)
    ally_s = sum(int(f[6]) for f in fleets if int(f[1]) == me)
    enemy_n = sum(1 for f in fleets if int(f[1]) != me)
    enemy_s = sum(int(f[6]) for f in fleets if int(f[1]) != me)
    turn = int(obs.get("step", 0))
    return np.array([
        sships/100.0, sprod/5.0, sr/4.0,
        tships/100.0, tprod/5.0, tr/4.0,
        own_self, own_neutral, own_enemy,
        ships_sent/100.0, ship_frac,
        dist/BOARD, eta/60.0, speed/MAX_SPEED,
        ally_n/10.0, ally_s/100.0, enemy_n/10.0, enemy_s/100.0,
        turn/500.0, my_ships_total/200.0, enemy_ships_total/200.0,
        (my_ships_total - enemy_ships_total)/200.0,
        my_planets/20.0, enemy_planets/20.0,
    ], dtype=np.float32)

def collect_one_game(args):
    """Run one v4 vs opponent game and return (features, labels) for every v4 shot."""
    teacher_path, opponent_path, seed, side, game_id = args
    paths = [teacher_path, opponent_path] if side == 0 else [opponent_path, teacher_path]
    env = make("orbit_wars", configuration={"randomSeed": seed}, debug=False)
    try:
        env.run(paths)
    except Exception as e:
        return [], game_id, str(e)
    rows = []
    for step_idx, st in enumerate(env.steps):
        s = st[side]
        obs = s.observation
        action = s.action or []
        if obs is None or not action: continue
        planets = obs["planets"]
        src_xy = {int(p[0]): (float(p[2]), float(p[3])) for p in planets}
        for mv in action:
            try:
                src_id, ang, ships = int(mv[0]), float(mv[1]), int(mv[2])
            except Exception:
                continue
            if src_id not in src_xy: continue
            tgt_id = find_target_via_ray(src_xy[src_id], ang, planets)
            if tgt_id < 0 or tgt_id == src_id: continue
            tgt_owner = next((int(p[1]) for p in planets if int(p[0]) == tgt_id), -2)
            if tgt_owner == side: continue  # skip own-planet reinforcements (trivially "successful")
            feat = encode_shot(obs, src_id, tgt_id, ships)
            if feat is None: continue
            tx, ty, tr = next(((float(p[2]), float(p[3]), float(p[4])) for p in planets if int(p[0]) == tgt_id), (0,0,0))
            sx, sy = src_xy[src_id]
            sr = next((float(p[4]) for p in planets if int(p[0]) == src_id), 0)
            dist = max(math.hypot(tx-sx, ty-sy) - sr - tr, 0.0)
            speed = fleet_speed(ships)
            eta_turns = max(int(math.ceil(dist / max(speed, 0.5))), 1)
            arrival_turn = step_idx + eta_turns
            label = label_outcome(env.steps, tgt_id, side, arrival_turn, window=10)
            rows.append((feat, label, game_id, step_idx))
    return rows, game_id, None

# Build job list: 5 opponents × 6 seeds × 2 sides = 60 games
TEACHER = str(V4_LOCAL)
SEEDS = list(range(101, 111))  # 10 seeds (was 6 in v1)
SELFPLAY_SEEDS = list(range(101, 131))  # 30 self-play seeds (was 24 in v1)
jobs = []
gid = 0
for opp in OPPONENT_PATHS:
    for seed in (SELFPLAY_SEEDS if opp == TEACHER else SEEDS):
        for side in (0, 1):
            gid += 1
            jobs.append((TEACHER, opp, seed, side, gid))
print(f"Jobs: {len(jobs)} games (teacher=v4, {len(OPPONENT_PATHS)} opps, {len(SEEDS)} seeds × 2 sides)")

In [ ]:
# Run jobs in parallel (Kaggle notebooks have ~4 CPU cores)
all_rows = []
failed = 0
t0 = time.time()
with mp.Pool(processes=4) as pool:
    for i, (rows, gid_, err) in enumerate(pool.imap_unordered(collect_one_game, jobs)):
        if err is not None:
            failed += 1
            print(f"  [WARN] game {gid_} failed: {err[:80]}")
        else:
            all_rows.extend(rows)
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(jobs)} games, rows={len(all_rows)}, t={time.time()-t0:.0f}s", flush=True)

print(f"\nDone: {len(all_rows)} shots collected ({failed} games failed)")
feats = np.stack([r[0] for r in all_rows]).astype(np.float32)
labels = np.asarray([r[1] for r in all_rows], dtype=np.float32)
meta_game = np.asarray([r[2] for r in all_rows], dtype=np.int32)
pos_rate = labels.mean()
print(f"  features: {feats.shape}, labels: {labels.shape}")
print(f"  positive rate: {pos_rate*100:.1f}%")

# Diagnostic: pos_rate is the most predictive number for whether the validator
# will produce a useful threshold. Healthy range is 50-75%. Above 85% means
# opponents are too weak (v4 dominates), training pos_weight will be too small,
# the validator will under-confidence almost every shot, and topk1 will end up
# sending almost nothing on the LB. If you see pos_rate > 85%, add stronger
# opponents or increase the v4-self-play seed count.
if pos_rate > 0.85:
    print(f"  WARNING: pos_rate is high ({pos_rate*100:.1f}%). Validator may be over-cautious.")
    print(f"           Consider stronger opponents or more v4 self-play games.")
elif pos_rate < 0.40:
    print(f"  WARNING: pos_rate is low ({pos_rate*100:.1f}%). Opponents may be too strong.")

# Save dataset for inspection
np.savez_compressed(WORK / "shot_dataset.npz", features=feats, labels=labels.astype(np.int64), meta_game=meta_game, feature_names=np.asarray(FEATURE_NAMES))
print(f"  saved {WORK}/shot_dataset.npz")

## 4. Train the ML shot validator

Current features are compact and submission-safe. Suggested feature improvements to test next:

- Add local pressure around source/target: allied/enemy ships within a fixed radius, nearest allied planet distance, nearest enemy planet distance.
- Add capture margin features: `ships_sent - (target_ships + target_prod * eta)`, and post-launch source reserve.
- Add global economy shares: my/enemy production totals, ship share, planet share, and target production share.
- Add angle-quality features: cosine/sine error between emitted angle and recovered target ray.

Network ideas worth testing next:

- Sweep hidden sizes 64, 96, and 128 before changing architecture.
- Try AdamW with light weight decay; it is cheap and keeps export unchanged.
- Keep the inference graph NumPy-friendly: Linear + ReLU + sigmoid is reliable for Kaggle submission.
- Tune validator threshold and top-k from game wins, not only row accuracy.


In [ ]:
import torch
import torch.nn as nn

class ShotValidator(nn.Module):
    def __init__(self, in_dim=FEATURE_DIM, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# Try CUDA, fall back to CPU on any compatibility issue.
# This MLP is tiny (3.5k params, 10k rows) so CPU is fine — full training
# completes in ~30 seconds either way.
def _select_device():
    if not torch.cuda.is_available():
        return torch.device("cpu"), "cuda not available"
    try:
        # Sanity-check that CUDA actually works on this device
        _ = torch.zeros(4, device="cuda") + 1
        torch.cuda.synchronize()
        return torch.device("cuda"), "cuda OK"
    except Exception as e:
        return torch.device("cpu"), f"cuda failed ({type(e).__name__}: {str(e)[:60]}), using CPU"

device, why = _select_device()
print(f"Training on: {device} ({why})")

# Game-level val split (random row split would leak — shots within a game are correlated)
rng = np.random.default_rng(42)
games = np.unique(meta_game)
rng.shuffle(games)
n_val = max(1, int(len(games) * 0.2))
val_games = set(games[:n_val].tolist())
val_mask = np.array([g in val_games for g in meta_game], dtype=bool)
Xt, yt = feats[~val_mask], labels[~val_mask]
Xv, yv = feats[val_mask], labels[val_mask]
print(f"  train: {len(Xt)} shots ({len(games)-n_val} games), val: {len(Xv)} shots ({n_val} games)")
print(f"  train pos: {yt.mean()*100:.1f}%, val pos: {yv.mean()*100:.1f}%")

# Class-balanced pos_weight
pr = max(yt.mean(), 1e-6)
pos_weight = torch.tensor([(1.0 - pr) / pr], device=device)
print(f"  pos_weight (neg/pos): {pos_weight.item():.3f}")

In [ ]:
# Training loop
model = ShotValidator(in_dim=FEATURE_DIM, hidden=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Xt_t = torch.from_numpy(Xt).to(device)
yt_t = torch.from_numpy(yt).to(device)
Xv_t = torch.from_numpy(Xv).to(device)
yv_t = torch.from_numpy(yv).to(device)

EPOCHS = 40
BATCH = 512

best_acc03 = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    idx = torch.randperm(len(Xt_t), device=device)
    losses = []
    for i in range(0, len(idx), BATCH):
        b = idx[i:i+BATCH]
        logits = model(Xt_t[b])
        loss = crit(logits, yt_t[b])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss))
    tl = float(np.mean(losses))
    model.eval()
    with torch.no_grad():
        vlogits = model(Xv_t)
        vloss = float(crit(vlogits, yv_t))
        vprob = torch.sigmoid(vlogits)
        v_acc05 = float(((vprob > 0.5).float() == yv_t).float().mean())
        v_acc03 = float(((vprob > 0.3).float() == yv_t).float().mean())
    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"epoch {epoch:3d} | t_loss={tl:.4f} v_loss={vloss:.4f} "
              f"v_acc(0.5)={v_acc05:.3f} v_acc(0.3)={v_acc03:.3f}")

print("\nTraining complete.")

## 4.1 Feature Importance

This cell uses permutation importance on the game-level validation split. Higher `delta_loss` means the validator relies more on that feature.


In [ ]:
# Permutation feature importance on validation split.
model.eval()
with torch.no_grad():
    base_logits = model(Xv_t)
    base_loss = float(crit(base_logits, yv_t))
    base_prob = torch.sigmoid(base_logits)
    base_acc03 = float(((base_prob > 0.3).float() == yv_t).float().mean())

rng = np.random.default_rng(123)
importance_rows = []
for j, name in enumerate(FEATURE_NAMES):
    Xp = Xv.copy()
    Xp[:, j] = rng.permutation(Xp[:, j])
    Xp_t = torch.from_numpy(Xp).to(device)
    with torch.no_grad():
        logits = model(Xp_t)
        loss = float(crit(logits, yv_t))
        prob = torch.sigmoid(logits)
        acc03 = float(((prob > 0.3).float() == yv_t).float().mean())
    importance_rows.append({
        "feature": name,
        "delta_loss": loss - base_loss,
        "delta_acc03": base_acc03 - acc03,
    })

importance_rows = sorted(importance_rows, key=lambda r: (r["delta_loss"], r["delta_acc03"]), reverse=True)
print(f"Baseline val loss={base_loss:.4f}, acc@0.3={base_acc03:.3f}")
print("Top permutation importances:")
for r in importance_rows[:20]:
    print(f"  {r['feature']:<24} delta_loss={r['delta_loss']:+.5f} delta_acc03={r['delta_acc03']:+.4f}")

try:
    import pandas as pd
    importance_df = pd.DataFrame(importance_rows)
    importance_df.to_csv(WORK / "feature_importance.csv", index=False)
    display(importance_df.head(20))
    print(f"saved {WORK}/feature_importance.csv")
except Exception as e:
    print(f"pandas/display unavailable ({type(e).__name__}); printed table above only")


## 5. Export weights to NumPy

The validator weights are exported to `weights.npz`. The wrapper runs validator inference with NumPy matmuls + ReLU + sigmoid; the base OrbitLite heuristic and its `orbit_lite/` helper package are bundled separately in the submission archive.


In [ ]:
model.eval()
sd = model.state_dict()
weights = {
    "l0_w": sd["net.0.weight"].cpu().numpy().astype(np.float32),
    "l0_b": sd["net.0.bias"].cpu().numpy().astype(np.float32),
    "l1_w": sd["net.2.weight"].cpu().numpy().astype(np.float32),
    "l1_b": sd["net.2.bias"].cpu().numpy().astype(np.float32),
    "l2_w": sd["net.4.weight"].cpu().numpy().astype(np.float32),
    "l2_b": sd["net.4.bias"].cpu().numpy().astype(np.float32),
}
for k, v in weights.items():
    print(f"  {k}: {v.shape}, {v.dtype}")

weights_path = WORK / "weights.npz"
np.savez(weights_path, **weights)
print(f"\nSaved {weights_path} ({weights_path.stat().st_size:,} bytes)")

# Sanity check: parity between PyTorch and numpy forward
def numpy_forward(x, W):
    h = np.maximum(0, x @ W["l0_w"].T + W["l0_b"])
    h = np.maximum(0, h @ W["l1_w"].T + W["l1_b"])
    logit = h @ W["l2_w"].T + W["l2_b"]
    return 1.0 / (1.0 + np.exp(-logit))

x_test = Xv[:5]
with torch.no_grad():
    p_torch = torch.sigmoid(model(torch.from_numpy(x_test).to(device))).cpu().numpy()
p_numpy = numpy_forward(x_test, weights).squeeze()
print(f"\nParity check (PyTorch vs numpy):")
print(f"  torch: {p_torch[:5]}")
print(f"  numpy: {p_numpy[:5]}")
print(f"  max diff: {np.abs(p_torch - p_numpy).max():.6e}")

## 6. Assemble `main.py`

We splice three components into a generated `main.py`:

1. The OrbitLite base source, with its `agent` function renamed to `_v4_agent_internal`.
2. A NumPy validator wrapper that filters low-confidence offensive shots.
3. A configurable top-k throttle; `main.py` uses topk1 and companion files `main_topk1.py`, `main_topk2.py`, `main_topk3.py` are created for experiments.


In [ ]:
# Read v4 source
v4_source = V4_LOCAL.read_text()
print(f"v4 source: {len(v4_source.splitlines()):,} lines, {len(v4_source):,} chars")

# Hybrid wrapper (renames v4's `agent` to `_v4_agent_internal` so we can wrap it)
import re
v4_renamed = re.sub(
    r"^def agent\(([^)]*)\):",
    r"def _v4_agent_internal(\1):",
    v4_source, count=1, flags=re.MULTILINE,
)
assert "_v4_agent_internal" in v4_renamed, "failed to rename v4 agent"

# Hybrid wrapper code (validator + topk1)
HYBRID_CODE = '''
# ============================================================
# ML Shot Validator (numpy) + topk1 throttle (v3: reverted from topk2)
# ============================================================
import os as _os_h
from pathlib import Path as _Path_h
import numpy as _np_h
import math as _math_h

_VAL_THRESHOLD = 0.30  # the "sweet spot" found via threshold sweep
_VAL_TOP_K = 1

def _find_weights():
    cands = [
        _Path_h("/kaggle_simulations/agent/weights.npz"),
        _Path_h.cwd() / "weights.npz",
        _Path_h("weights.npz"),
    ]
    try:
        cands.insert(0, _Path_h(__file__).resolve().parent / "weights.npz")
    except NameError:
        pass
    for p in cands:
        if p.exists(): return p
    return None

_W_PATH = _find_weights()
_W = _np_h.load(_W_PATH) if _W_PATH is not None else None

def _validator_proba(x):
    if _W is None: return None
    h = _np_h.maximum(0, x @ _W["l0_w"].T + _W["l0_b"])
    h = _np_h.maximum(0, h @ _W["l1_w"].T + _W["l1_b"])
    logit = h @ _W["l2_w"].T + _W["l2_b"]
    return 1.0 / (1.0 + _np_h.exp(-logit))

_BOARD_H = 100.0; _MAX_SPEED_H = 6.0

def _fleet_speed_h(ships):
    if ships <= 0: return 1.0
    return 1.0 + (_MAX_SPEED_H - 1.0) * (_math_h.log(max(ships, 1)) / _math_h.log(1000.0)) ** 1.5

def _find_target_ray_h(src_xy, ang, planets, ray_horizon=200.0, perp_margin=1.0):
    sx, sy = src_xy; fx, fy = _math_h.cos(ang), _math_h.sin(ang)
    best_pid, best_perp = -1, 1e9
    for p in planets:
        pid, _, px, py, pr, _, _ = p
        pid = int(pid); px = float(px); py = float(py); pr = float(pr)
        dx = px - sx; dy = py - sy
        t = dx * fx + dy * fy
        if t <= 0 or t > ray_horizon: continue
        perp = abs(dx * fy - dy * fx)
        if perp <= pr + perp_margin and perp < best_perp:
            best_perp = perp; best_pid = pid
    return best_pid

def _encode_shot_h(obs, src_id, target_id, ships_sent):
    pdict = {int(p[0]): p for p in obs["planets"]}
    if src_id not in pdict or target_id not in pdict: return None
    src = pdict[src_id]; tgt = pdict[target_id]
    me = int(obs.get("player", 0))
    fleets = obs.get("fleets", [])
    planets = obs["planets"]
    my_t = sum(int(p[5]) for p in planets if int(p[1]) == me)
    en_t = sum(int(p[5]) for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    my_p = sum(1 for p in planets if int(p[1]) == me)
    en_p = sum(1 for p in planets if int(p[1]) >= 0 and int(p[1]) != me)
    sx, sy, sr, sships = float(src[2]), float(src[3]), float(src[4]), int(src[5])
    tx, ty, tr, tships = float(tgt[2]), float(tgt[3]), float(tgt[4]), int(tgt[5])
    sprod, tprod = float(src[6]), float(tgt[6])
    dx, dy = tx - sx, ty - sy
    dist = max(_math_h.hypot(dx, dy) - sr - tr, 0.0)
    speed = _fleet_speed_h(ships_sent); eta = dist / max(speed, 0.5)
    own_self = 1.0 if int(tgt[1]) == me else 0.0
    own_neutral = 1.0 if int(tgt[1]) < 0 else 0.0
    own_enemy = 1.0 if (int(tgt[1]) >= 0 and int(tgt[1]) != me) else 0.0
    sf = ships_sent / max(sships, 1)
    an = sum(1 for f in fleets if int(f[1]) == me)
    a_s = sum(int(f[6]) for f in fleets if int(f[1]) == me)
    en = sum(1 for f in fleets if int(f[1]) != me)
    e_s = sum(int(f[6]) for f in fleets if int(f[1]) != me)
    turn = int(obs.get("step", 0))
    return _np_h.array([
        sships/100.0, sprod/5.0, sr/4.0,
        tships/100.0, tprod/5.0, tr/4.0,
        own_self, own_neutral, own_enemy,
        ships_sent/100.0, sf,
        dist/_BOARD_H, eta/60.0, speed/_MAX_SPEED_H,
        an/10.0, a_s/100.0, en/10.0, e_s/100.0,
        turn/500.0, my_t/200.0, en_t/200.0,
        (my_t - en_t)/200.0,
        my_p/20.0, en_p/20.0,
    ], dtype=_np_h.float32)

def agent(obs, config=None):
    try:
        moves = _v4_agent_internal(obs, config)
    except TypeError:
        moves = _v4_agent_internal(obs)
    if not moves or _W is None:
        # If validator missing or v4 didn't propose anything, return raw v4 output
        return moves
    side = int(obs.get("player", 0))
    planets = obs["planets"]
    src_xy = {int(p[0]): (float(p[2]), float(p[3])) for p in planets}
    owner_by_id = {int(p[0]): int(p[1]) for p in planets}
    feats, idxs = [], []
    for i, mv in enumerate(moves):
        try:
            sid, ang, ships = int(mv[0]), float(mv[1]), int(mv[2])
        except Exception:
            continue
        if sid not in src_xy: continue
        tid = _find_target_ray_h(src_xy[sid], ang, planets)
        if tid < 0 or tid == sid: continue
        if owner_by_id.get(tid, -2) == side:
            continue  # never filter own-planet reinforcements
        f = _encode_shot_h(obs, sid, tid, ships)
        if f is None: continue
        feats.append(f); idxs.append(i)
    if feats:
        x = _np_h.stack(feats)
        probs = _validator_proba(x).squeeze(-1)
        keep = [True] * len(moves)
        for j, p in zip(idxs, probs):
            if p < _VAL_THRESHOLD:
                keep[j] = False
        moves = [m for k, m in enumerate(moves) if keep[k]]
    if not moves: return []
    # Keep only the largest ship-count moves. Change _VAL_TOP_K for topk sweeps.
    if _VAL_TOP_K and _VAL_TOP_K > 0 and len(moves) > _VAL_TOP_K:
        ranked = sorted(enumerate(moves), key=lambda im: int(im[1][2]), reverse=True)[:_VAL_TOP_K]
        keep_idx = {i for i, _ in ranked}
        moves = [m for i, m in enumerate(moves) if i in keep_idx]
    return moves
'''

# Concatenate v4 (renamed) + hybrid wrapper
main_py = v4_renamed + "\n\n" + HYBRID_CODE

# Write to /kaggle/working/main.py — Kaggle picks up `agent` automatically
(WORK / "main.py").write_text(main_py)
print(f"main.py: {(WORK/'main.py').stat().st_size:,} bytes ({len(main_py.splitlines()):,} lines)")

# Build top-k variants for local experiments. main.py remains topk1.
TOPK_AGENT_PATHS = {}
for _k in (1, 2, 3):
    _variant = main_py.replace("_VAL_TOP_K = 1", f"_VAL_TOP_K = {_k}")
    _path = WORK / f"main_topk{_k}.py"
    _path.write_text(_variant)
    TOPK_AGENT_PATHS[_k] = _path
    print(f"{_path.name}: {_path.stat().st_size:,} bytes")
FINAL_AGENT_PATH = WORK / "main.py" 

## 7. Sanity check

Run a single game with the freshly-built `main.py` to make sure it loads, runs, and finishes without errors.

In [ ]:
# Sanity test: run a 1-game match to confirm the agent works
from kaggle_environments import make

env = make("orbit_wars", debug=True)
result = env.run([str(FINAL_AGENT_PATH), OPPONENT_PATHS[0]])
final = env.steps[-1]
print(f"Game finished in {len(env.steps)} steps:")
for i, s in enumerate(final):
    print(f"  P{i}: reward={s.reward}, status={s.status}")

## 8. Top-K Experiment

This optional cell compares validator throttles `topk1`, `topk2`, and `topk3` against a small opponent/seed sample. Increase seeds and opponents for a more reliable decision.


In [ ]:
import statistics
from kaggle_environments import make

def slot_metrics(env, slot):
    total_actions = 0
    active_turns = 0
    observable_turns = 0
    statuses = {}
    for step in env.steps:
        state = step[slot]
        statuses[state.status] = statuses.get(state.status, 0) + 1
        if state.observation is None:
            continue
        observable_turns += 1
        actions = state.action or []
        total_actions += len(actions)
        active_turns += int(bool(actions))
    reward = float(env.steps[-1][slot].reward or 0.0)
    return {
        "reward": reward,
        "steps": float(len(env.steps)),
        "moves_per_turn": total_actions / max(1, observable_turns),
        "idle_rate": 1.0 - active_turns / max(1, observable_turns),
        "invalid_turns": float(statuses.get("INVALID", 0)),
        "error_turns": float(statuses.get("ERROR", 0)),
        "timeout_turns": float(statuses.get("TIMEOUT", 0)),
    }

def summarize_rows(rows, label_key="agent"):
    for name in sorted({r[label_key] for r in rows}):
        selected = [r for r in rows if r[label_key] == name]
        wins = sum(float(r["reward"]) > 0 for r in selected)
        losses = sum(float(r["reward"]) < 0 for r in selected)
        draws = len(selected) - wins - losses
        print(
            f"{name}: W-L-D={wins}-{losses}-{draws} "
            f"win_rate={wins / max(1, len(selected)):.3f} "
            f"avg_steps={statistics.fmean(float(r['steps']) for r in selected):.1f} "
            f"moves/t={statistics.fmean(float(r['moves_per_turn']) for r in selected):.3f} "
            f"idle={statistics.fmean(float(r['idle_rate']) for r in selected):.3f} "
            f"invalid={sum(float(r['invalid_turns']) for r in selected):.0f} "
            f"errors={sum(float(r['error_turns']) for r in selected):.0f} "
            f"timeouts={sum(float(r['timeout_turns']) for r in selected):.0f}"
        )

RUN_TOPK_SWEEP = True
SWEEP_SEEDS = list(range(501, 503))
SWEEP_OPPONENTS = OPPONENT_PATHS[:min(2, len(OPPONENT_PATHS))]

topk_rows = []
if RUN_TOPK_SWEEP:
    for top_k, agent_path in TOPK_AGENT_PATHS.items():
        for opp in SWEEP_OPPONENTS:
            for seed in SWEEP_SEEDS:
                for labels in ((f"topk{top_k}", "opp"), ("opp", f"topk{top_k}")):
                    paths = [str(agent_path), opp] if labels[0].startswith("topk") else [opp, str(agent_path)]
                    env = make("orbit_wars", debug=False, configuration={"randomSeed": seed})
                    env.run(paths)
                    for slot, name in enumerate(labels):
                        if not name.startswith("topk"):
                            continue
                        row = {"top_k": top_k, "agent": name, "opponent": Path(opp).name, "seed": seed, "seat": slot}
                        row.update(slot_metrics(env, slot))
                        topk_rows.append(row)
                    print(f"topk={top_k} opp={Path(opp).name} seed={seed} seats={labels[0]}/{labels[1]} steps={len(env.steps)}", flush=True)
    summarize_rows(topk_rows)
else:
    print("RUN_TOPK_SWEEP=False; set it to True to run the experiment.")


## 9. Pure Heuristic vs Validator Benchmark

This mirrors `benchmark_buddy_vs_h3b1.py`: both agents play both seats across a seed range, then the summary reports W-L-D plus action/status diagnostics.


In [ ]:
import importlib.util
import sys
from pathlib import Path
from kaggle_environments import make

def load_agent(path, module_name):
    path = Path(path)
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import agent from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module.agent

def benchmark_pair(seed_start=901, seeds=5):
    agents = {
        "heuristic": load_agent(V4_LOCAL, "bench_orbitlite_heuristic"),
        "validator": load_agent(FINAL_AGENT_PATH, "bench_orbitlite_validator"),
    }
    rows = []
    for seed in range(seed_start, seed_start + seeds):
        for labels in (("heuristic", "validator"), ("validator", "heuristic")):
            env = make("orbit_wars", debug=False, configuration={"randomSeed": seed})
            env.run([agents[labels[0]], agents[labels[1]]])
            for slot, name in enumerate(labels):
                row = {"seed": seed, "seat": slot, "agent": name}
                row.update(slot_metrics(env, slot))
                rows.append(row)
            winner = max(range(2), key=lambda slot: float(env.steps[-1][slot].reward or 0.0))
            print(f"seed={seed} seats={labels[0]}/{labels[1]} winner={labels[winner]} steps={len(env.steps)}", flush=True)
    return rows

RUN_HEAD_TO_HEAD_BENCH = True
if RUN_HEAD_TO_HEAD_BENCH:
    bench_rows = benchmark_pair(seed_start=901, seeds=5)
    summarize_rows(bench_rows)
else:
    print("RUN_HEAD_TO_HEAD_BENCH=False; set it to True to run the benchmark.")


## 10. Build `submission.tar.gz`

Bundle `main.py`, `weights.npz`, and the `orbit_lite/` helper package into the tar.gz format Kaggle expects.


In [ ]:
import tarfile

tar_path = WORK / "submission.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(FINAL_AGENT_PATH, arcname="main.py")
    tar.add(WORK / "weights.npz", arcname="weights.npz")
    for helper_file in sorted((WORK / "orbit_lite").glob("*.py")):
        tar.add(helper_file, arcname=f"orbit_lite/{helper_file.name}")

size = tar_path.stat().st_size
print(f"\nSubmission ready: {tar_path} ({size:,} bytes = {size/1024:.1f} KB)")
with tarfile.open(tar_path) as tar:
    for m in tar.getmembers():
        print(f"  {m.name}: {m.size:,} bytes")


## 11. How to submit

Two options:

### A. Submit this notebook directly (recommended)

1. Click **Save Version** at the top right of the notebook → "Save & Run All (Commit)"
2. From the notebook page, click **Submit to Competition** on the right → choose `submission.tar.gz`

Kaggle automatically picks up `/kaggle/working/submission.tar.gz`.

### B. Download and submit via CLI

1. Download `submission.tar.gz` from the notebook output
2. Locally:
   ```bash
   kaggle competitions submit orbit-wars -f submission.tar.gz -m "v4 + ML validator (t=0.30) + topk1"
   ```

## 10. Expected score

- **LB ~970 ballpark** with v3 (topk1 + the v2 data improvements). The previous v2 attempt with topk2 scored 841.8; the original v1 (no defender opp, 6 seeds) was 926.5. v3 should land **above v1** thanks to the additional training data, while reverting the throttle mistake from v2.
- The opponent pool here is a self-contained set of **simple** inline agents, weaker than the original `exp_arch_001` pool. With stronger external opponents (e.g. `orbit-botnext`), scores tend to be higher.
- To push higher, try:
  - **Threshold sweep**: 0.25 / 0.30 / 0.35 around the current default. Looser (0.25) keeps more shots; tighter (0.35) drops more — pick by validation accuracy or local win rate.
  - **Stronger opponents in self-play**: include the public `orbit-botnext` notebook by Pascal Ledesma — adds noticeable LB pts in our experiments.
  - **More data**: scale to 200+ games (vs current ~140).
  - **Reconsider topk2**: only after you have a stronger validator (e.g. trained on real LB-≥1450 player replays). With this notebook's simple-opponent training, topk1 is better — see section 6 sweep.

## 11. Improvement ideas

1. **Train against stronger opponents** — the agent's known weakness is matchups vs LB-≥1000 opponents (~20% win rate). Including stronger agents in the data-collection pool should help directly.
2. **Richer features** — add comet info, planet orbital velocity, and rotation phase to the 24-dim feature vector.
3. **Validator ensemble** — train one MLP on the v1 dataset and one on a hard-negatives augmented dataset, accept a shot only if both approve.
4. **Dynamic threshold** — vary `_VAL_THRESHOLD` by game phase (stricter early, looser late, or vice versa).

## Final acknowledgments

This notebook stands on the shoulders of the public Kaggle community. Particular thanks to:
- **Pascal Ledesma** for [orbit-botnext](https://www.kaggle.com/code/pascalledesma/orbit-botnext) and [orbitwork-v14](https://www.kaggle.com/code/pascalledesma/orbitwork-v14)
- **djenkivanov** for [orbit-wars-agent-ow-proto](https://www.kaggle.com/code/djenkivanov/orbit-wars-agent-ow-proto)
- The authors of the LB-1224 / LB-1100 public notebooks whose constants we incorporated

If this notebook helps you, please give an upvote to **their** notebooks too — without their work this hybrid would not exist.

In [ ]:
# Optional Kaggle CLI submit. Usually you can submit from the notebook UI after Save & Run All.
!kaggle competitions submit orbit-wars -f submission.tar.gz -m "orbitlite heuristic + ml validator"
